# Different Models Differentiate Neighbours


In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import AzureOpenAI

# Load variables from .env in the workspace root
load_dotenv(dotenv_path=Path('.env'))

def required_env(name: str) -> str:
    value = os.getenv(name, '').strip()
    if not value:
        raise ValueError(f"Missing required environment variable: {name}")
    return value

# Azure OpenAI embedding deployment.
# OpenAI has no "text-embedding-ada-003" model. Use an Azure deployment of
# text-embedding-ada-002 or text-embedding-3-small and put its deployment name
# in AZURE_OPENAI_EMBD_MODEL.
embed_client = AzureOpenAI(
    api_key=required_env('AZURE_OPENAI_API_KEY'),
    azure_endpoint=required_env('AZURE_OPENAI_ENDPOINT'),
    api_version=os.getenv('AZURE_OPENAI_API_VERSION', '2024-02-15-preview').strip() or '2024-02-15-preview',
)
emb_model = os.getenv('AZURE_OPENAI_EMBD_MODEL', 'text-embedding-ada-002').strip() or 'text-embedding-ada-002'

In [2]:
import gensim.downloader as api
import numpy as np

# -------------------------------------------------
# 1. Load a pretrained STATIC word embedding model
# -------------------------------------------------
# This downloads the model the first time you run it.
# It is relatively small and good for demos.
model = api.load("glove-wiki-gigaword-50")

# -------------------------------------------------
# 2. Tokenize a word (very simple tokenization)
# -------------------------------------------------
def tokenize_word(word: str) -> str:
    return word.lower().strip()

# -------------------------------------------------
# 3. Get embedding vector for a token
# -------------------------------------------------
def get_embedding(token: str):
    if token in model:
        return model[token]
    else:
        return None
# -------------------------------------------------

In [3]:
# -------------------------------------------------
# 4. Compare nearest neighbours from two embedding models
# -------------------------------------------------
def glove_semantic_search(token: str, candidates: list[str], topn: int = 10):
    """Search with GloVe over the supplied candidate words."""
    if token not in model:
        return []

    candidate_vectors = np.asarray([model[word] for word in candidates])
    query_vector = model[token]
    scores = candidate_vectors @ query_vector / (
        np.linalg.norm(candidate_vectors, axis=1) * np.linalg.norm(query_vector)
    )
    best_indices = np.argsort(scores)[-topn:][::-1]
    return [(candidates[index], float(scores[index])) for index in best_indices]


def openai_semantic_search(
    token: str,
    candidates: list[str],
    topn: int = 10,
    batch_size: int = 512,
):
    """Embed and rank the texts, returning neighbours and the vector dimension."""
    texts = [token, *candidates]
    vectors = []

    for start in range(0, len(texts), batch_size):
        response = embed_client.embeddings.create(
            model=emb_model,
            input=texts[start:start + batch_size],
        )
        vectors.extend(item.embedding for item in response.data)

    matrix = np.asarray(vectors, dtype=np.float32)
    dimensions = matrix.shape[1]
    matrix /= np.linalg.norm(matrix, axis=1, keepdims=True)
    scores = matrix[1:] @ matrix[0]
    best_indices = np.argsort(scores)[-topn:][::-1]
    neighbours = [(candidates[index], float(scores[index])) for index in best_indices]
    return neighbours, dimensions

In [4]:
query_word = "bank"
topn_n = 10
candidate_count = 5_000
token = tokenize_word(query_word)

# Use the same candidate vocabulary for both models so the comparison is fair.
candidates = [
    word for word in model.index_to_key[:candidate_count]
    if word != token
]

print("-" * 96)
print("Nearest-Neighbour comparison")
print(f"Token: {token!r}")
#print(f"Candidates: {len(candidates):,} common GloVe vocabulary words")
print(f"Models: GloVe and Azure OpenAI deployment {emb_model!r}")
print("-" * 96)

if token not in model:
    print(f"Token {token!r} not found in the GloVe vocabulary.")
else:
    glove_neighbours = glove_semantic_search(token, candidates, topn=topn_n)
    openai_neighbours, openai_dimensions = openai_semantic_search(
        token,
        candidates,
        topn=topn_n,
    )

    glove_heading = f"GloVe neighbour ({model.vector_size}d)"
    openai_heading = f"OpenAI neighbour ({openai_dimensions}d)"
    print(f"{glove_heading:27s} {'score':>8s} | {openai_heading:30s} {'score':>8s}")
    print("-" * 96)
    for (glove_word, glove_score), (openai_word, openai_score) in zip(
        glove_neighbours,
        openai_neighbours,
    ):
        print(
            f"{glove_word:27s} {glove_score:8.4f} | "
            f"{openai_word:30s} {openai_score:8.4f}"
        )

    glove_words = {word for word, _ in glove_neighbours}
    openai_words = {word for word, _ in openai_neighbours}
    overlap = glove_words & openai_words
    print("-" * 96)
    print(f"Shared neighbours: {len(overlap)}/{topn_n} ({', '.join(sorted(overlap)) or 'none'})")
    print("Different rankings are expected because the models were trained differently.")

------------------------------------------------------------------------------------------------
Nearest-Neighbour comparison
Token: 'bank'
Models: GloVe and Azure OpenAI deployment 'text-embedding-ada-002'
------------------------------------------------------------------------------------------------
GloVe neighbour (50d)          score | OpenAI neighbour (1536d)          score
------------------------------------------------------------------------------------------------
banks                         0.8699 | banks                            0.9405
securities                    0.7997 | bond                             0.8989
banking                       0.7965 | banking                          0.8953
investment                    0.7850 | loan                             0.8869
exchange                      0.7809 | finance                          0.8787
financial                     0.7670 | court                            0.8784
credit                        0.7649 | branch 